In [92]:
!pip install category_encoders

In [93]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (mean_absolute_error,mean_absolute_percentage_error,mean_squared_error,r2_score)

import category_encoders as ce

In [94]:
train_url = ( "https://raw.githubusercontent.com/goitacademy/" "MACHINE-LEARNING-NEO/main/datasets/mod_04_hw_train_data.csv" )
valid_url = ( "https://raw.githubusercontent.com/goitacademy/" "MACHINE-LEARNING-NEO/main/datasets/mod_04_hw_valid_data.csv" )
df = pd.read_csv(train_url)
tf = pd.read_csv(valid_url)

In [95]:
print(f"Розмір тренувального набору: {df.shape}")
print(f"Розмір валідаційного набору: {tf.shape}")

Розмір тренувального набору: (249, 9)
Розмір валідаційного набору: (7, 9)


In [96]:
df.head()

,Name,Phone_Number,Experience,Qualification,University,Role,Cert,Date_Of_Birth,Salary
0,Jennifer Hernandez,120-602-1220,3.0,Msc,Tier2,Mid,Yes,25/08/1972,98000
1,Timothy Walker,840-675-8650,5.0,PhD,Tier2,Senior,Yes,03/12/2013,135500
2,David Duran,556-293-8643,5.0,Msc,Tier2,Senior,Yes,19/07/2002,123500
3,Gloria Ortega,463-559-7474,3.0,Bsc,Tier3,Mid,No,19/02/1970,85000
4,Matthew Steele,968-091-7683,5.0,Bsc,Tier2,Senior,Yes,20/02/1970,111500


In [97]:
tf

,Name,Phone_Number,Experience,Qualification,University,Role,Cert,Date_Of_Birth,Salary
0,Alvaro Johnson,320-636-8883,7,Bsc,Tier1,Senior,No,12/03/1978,109300
1,Austin Powers,903-121-1691,2,Msc,Tier1,Mid,Yes,13/03/1992,84800
2,Joshua Phil,673-972-2453,3,Bsc,Tier3,Mid,Yes,19/02/1988,98900
3,Mirinda Collins,310-364-6925,5,Msc,Tier2,Senior,No,20/03/1989,116500
4,Mustapha Green,401-249-3912,3,PhD,Tier1,Junior,Yes,21/03/1979,75800
5,Nick Freeman,875-546-2104,6,Bsc,Tier3,Junior,Yes,22/03/1982,97300
6,Pamela Allison,408-955-5085,2,PhD,Tier2,Junior,No,23/03/1968,69800


In [98]:
print("Колонки повністю збігаються:")
print(df.columns.tolist() == tf.columns.tolist())

Колонки повністю збігаються:
True


3. Первинний дослідницький аналіз даних

In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Name           249 non-null    object 
 1   Phone_Number   249 non-null    object 
 2   Experience     247 non-null    float64
 3   Qualification  248 non-null    object 
 4   University     249 non-null    object 
 5   Role           246 non-null    object 
 6   Cert           247 non-null    object 
 7   Date_Of_Birth  249 non-null    object 
 8   Salary         249 non-null    int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 17.6+ KB


In [100]:
df.duplicated().sum()

np.int64(0)

In [101]:
df.isnull().sum()

,0
Name,0
Phone_Number,0
Experience,2
Qualification,1
University,0
Role,3
Cert,2
Date_Of_Birth,0
Salary,0


In [102]:
columns_to_drop = ["Name", "Phone_Number", "Date_Of_Birth"]

df = df.drop(columns=columns_to_drop)
tf = tf.drop(columns=columns_to_drop)
print(df.columns)
print(tf.columns)

Index(['Experience', 'Qualification', 'University', 'Role', 'Cert', 'Salary'], dtype='object')
Index(['Experience', 'Qualification', 'University', 'Role', 'Cert', 'Salary'], dtype='object')


In [103]:
before = len(df)
df = df.dropna()
after = len(df)

print(f"Було: {before}")
print(f"Стало: {after}")
print(f"Видалено: {before-after}")

Було: 249
Стало: 241
Видалено: 8


In [104]:
# Розділяємо на числові та категоріальні колонки
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include='object').columns

print(num_cols)
print(cat_cols)

Index(['Experience', 'Salary'], dtype='object')
Index(['Qualification', 'University', 'Role', 'Cert'], dtype='object')


In [105]:
X_train = df.drop(columns="Salary")
y_train = df["Salary"]

X_test = tf.drop(columns="Salary")
y_test = tf["Salary"]

In [106]:
# Ініціалізація енкодера
encoder = ce.TargetEncoder()

# Навчання енкодера та трансформація тренувальних даних
X_train = encoder.fit_transform(X_train, y_train)

In [107]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 241 entries, 0 to 248
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Experience     241 non-null    float64
 1   Qualification  241 non-null    float64
 2   University     241 non-null    float64
 3   Role           241 non-null    float64
 4   Cert           241 non-null    float64
dtypes: float64(5)
memory usage: 11.3 KB


In [108]:
X_train.nunique()

,0
Experience,5
Qualification,3
University,3
Role,3
Cert,2


In [109]:
# Ініціалізація стандартного масштабування
scaler = StandardScaler().set_output(transform='pandas')

X_train = scaler.fit_transform(X_train)

In [110]:
# Показуємо наскільки розподіл даних відхиляється від симетричного (нормального).
X_train.skew()

,0
Experience,-0.373363
Qualification,0.252668
University,-0.490365
Role,-0.480976
Cert,-0.075210


In [111]:
# Трансформація тестових даних
X_test = encoder.transform(X_test)
# Нормалізація
X_test = scaler.transform(X_test)

In [112]:
knn_r_mod = KNeighborsRegressor(n_neighbors=13, weights="distance").fit(X_train, y_train)

y_pred = knn_r_mod.predict(X_test)

In [113]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = mean_absolute_percentage_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
r_sq = knn_r_mod.score(X_train, y_train)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2%}")
print(f"R²  : {r2:.3f}")
print(f'R2 train: {r_sq:.2f}')

MAE : 5862.61
RMSE: 7394.45
MAPE: 7.29%
R²  : 0.784
R2 train: 1.00


**Висновок:**

Під час попередньої обробки даних було проаналізовано наявність пропущених значень. Оскільки кількість рядків із пропущеними даними становила незначну частку від загального обсягу вибірки, було прийнято рішення видалити такі записи, а не застосовувати методи заповнення пропусків (імпутації).

Им'я людини та номер телефону видалені бо не мають впливу на результат моделі.

Окремо було досліджено вплив ознаки, отриманої з дати народження працівника. Для цього з поля Date_Of_Birth було виділено рік народження (Year) і використано його як додаткову ознаку під час навчання моделі.

Порівняння результатів показало, що включення року народження не призвело до покращення якості прогнозування заробітної плати. Навпаки, у проведених експериментах спостерігалося незначне погіршення значень метрик якості. Це свідчить про те, що для цього набору даних рік народження не є достатньо інформативною ознакою для прогнозування рівня заробітної плати.

Під час дослідження було порівняно декілька варіантів підготовки даних та налаштування моделі KNeighborsRegressor.

На етапі попередньої обробки було протестовано два методи масштабування числових ознак: `PowerTransformer` та `StandardScaler`. Експеримент показав, що використання `StandardScaler` забезпечує кращу якість прогнозування. Це можна пояснити тим, що більшість числових ознак у наборі даних не мали значної асиметрії розподілу, а категоріальні ознаки після кодування `TargetEncoder` містили лише невелику кількість унікальних значень. У такій ситуації додаткове перетворення розподілу за допомогою `PowerTransformer` не покращувало модель, тоді як `StandardScaler` ефективно привів усі ознаки до єдиного масштабу, що є особливо важливим для алгоритму KNN, який використовує евклідову відстань між об'єктами.

Також було проведено порівняння двох методів кодування категоріальних ознак (`OneHotEncoder` та `TargetEncoder`), різних значень параметра `n_neighbors` і способів зважування сусідів (`uniform` та `distance`).

Результати показали, що використання `weights="distance"` забезпечує кращу якість прогнозування порівняно з рівномірним зважуванням, оскільки найближчі сусіди мають більший вплив на прогноз.

Крім того, було встановлено, що збільшення кількості сусідів з 9 до 13 покращує якість моделі, однак подальше збільшення до 15 і 17 призводить до зниження значення R². Це свідчить про те, що надмірне збільшення кількості сусідів викликає надмірне усереднення прогнозів і втрату інформації про локальну структуру даних.

Найкращий результат було отримано для моделі `KNeighborsRegressor` з параметрами `n_neighbors=13`, `weights="distance"` та використанням `TargetEncoder` разом зі `StandardScaler`. На тестовому наборі модель досягла значення **R² = 0.784**, що є найкращим серед усіх протестованих конфігурацій. Проведені експерименти показали, що саме така комбінація методів підготовки даних і параметрів моделі найкраще описує залежність між характеристиками працівників та рівнем їхньої заробітної плати для цього набору даних.